In [20]:
# ============================================================
# MEMBER 3: Baseline Data Cleaning Pipeline
# ============================================================
import pandas as pd
import time, psutil, os

from google.colab import drive
drive.mount('/content/drive')

# --- LOAD ---
df = pd.read_csv('/content/drive/MyDrive/bernama_baseline_raw.csv')
print(f"Loaded: {len(df)} records")

# --- START BENCHMARK ---
process = psutil.Process(os.getpid())
start_time = time.time()
mem_before = process.memory_info().rss / (1024 * 1024)

# --- CLEAN ---
# 1. Remove duplicates by URL
df.drop_duplicates(subset='url', inplace=True)

# 2. Strip whitespace
for col in ['headline', 'section', 'article_summary', 'url']:
    df[col] = df[col].str.strip()

# 3. Standardize date
df['publication_date'] = pd.to_datetime(df['publication_date'], errors='coerce')
df.dropna(subset=['publication_date'], inplace=True)
df['publication_date'] = df['publication_date'].dt.strftime('%Y-%m-%d %H:%M')

# 4. Validate URL
df = df[df['url'].str.startswith('https://www.bernama.com')]

# 5. Filter short summaries
df = df[df['article_summary'].str.len() >= 40]

# 6. Standardize section casing
df['section'] = df['section'].str.title()

# 7. Add derived columns
df['pub_date_only'] = pd.to_datetime(df['publication_date']).dt.date
df['pub_year'] = pd.to_datetime(df['publication_date']).dt.year
df['pub_month'] = pd.to_datetime(df['publication_date']).dt.month
df['headline_word_count'] = df['headline'].str.split().str.len()

# --- END BENCHMARK ---
end_time = time.time()
mem_after = process.memory_info().rss / (1024 * 1024)
total_time = end_time - start_time
throughput = len(df) / total_time

print(f"\n✅ Final cleaned records: {len(df)}")
print(f"⏱️ Total Time: {total_time:.2f}s")
print(f"🚀 Throughput: {throughput:.0f} records/sec")
print(f"💾 Memory Used: {mem_after - mem_before:.1f} MB")

# --- SAVE ---
df.to_csv('/content/drive/MyDrive/cleaned_data.csv', index=False)

perf_before = pd.DataFrame([{
    'technique': 'Baseline (Single-threaded)',
    'total_time_sec': total_time,
    'records_processed': len(df),
    'throughput_rec_per_sec': throughput,
    'memory_mb': mem_after - mem_before,
}])
perf_before.to_csv('/content/drive/MyDrive/performance_before.csv', index=False)

print("\n✅ Saved: cleaned_data.csv")
print("✅ Saved: performance_before.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded: 101212 records

✅ Final cleaned records: 101212
⏱️ Total Time: 1.30s
🚀 Throughput: 77740 records/sec
💾 Memory Used: -6.0 MB

✅ Saved: cleaned_data.csv
✅ Saved: performance_before.csv
